<a href="https://colab.research.google.com/github/Xatuhan/Borsa/blob/optimizasyon/Borsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BİST Sıfır Güven Tarayıcısı (v7)

**Strateji (değişmedi, v6 ile aynı):** XU100 bileşenlerinin 1 yıllık günlük kapanışları üzerinden

| Filtre | Kural |
|---|---|
| 1. Ölüm kesişimi kalkanı | 50 SMA > 200 SMA |
| 2. Ana trend onayı | Son fiyat ≥ 50 SMA |
| 3. Pusu marjı | (Fiyat − 50 SMA) / 50 SMA ≤ %3 |
| Veri yeterliliği | En az 200 geçerli gün |

**Hücre sırası:** kurulum → ayarlar ve motor → tarama → seçilenlerin grafikleri → tek hisse inceleme → KAP haberleri.
"Run All" ile baştan sona çalışır. Veriler 15 dakika gecikmelidir.

v1–v6 arası eski deneme hücreleri aynı stratejiyi tekrar tekrar taradığı için çıkarıldı;
hepsi GitHub'da `main` dalının geçmişinde duruyor.

In [ ]:
# %pip, paketi not defterinin kendi çekirdeğine kurar (!pip bazen başka bir Python'a kurar).
# pandas / numpy / matplotlib / yfinance Colab'de hazır gelir; yoksa kurulur, varsa dokunulmaz.
%pip install -q borsapy yfinance pandas numpy matplotlib

In [ ]:
import logging
import random
import sys
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from typing import Callable, Optional, Tuple, TypeVar
from zoneinfo import ZoneInfo

import borsapy as bp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

# =====================================================================
# AYARLAR — STRATEJİ EŞİKLERİ (v6 ile birebir aynı)
# =====================================================================
ENDEKS = "XU100"
SMA_KISA = 50
SMA_UZUN = 200
MAKS_UZAKLIK = 0.03      # Fiyat, 50 SMA'nın en fazla %3 üstünde olabilir
MIN_VERI_GUNU = 200

# --- Ağ ayarları (strateji değil, yalnızca veri çekme davranışı) ---
# TradingView kısa sürede çok bağlantı açan IP'ye 429 (Too Many Requests) verir.
# Bu yüzden istekler "insan gibi" aralıklı ve düzensiz açılır; hız sunucunun izin verdiği kadardır.
VERI_PERIYODU = "1y"
SAATLIK_PERIYOT = "1mo"
PARALEL_ISTEK = 2         # Aynı anda en fazla 2 bağlantı (yalnızca yavaş yanıtları örtmek için)
TV_ISTEK_ARALIGI = 1.0    # İki yeni TradingView bağlantısı arasında en az 1 sn ...
TV_SAPMA = 0.8            # ... + 0–0.8 sn rastgele (sabit ritim bot gibi görünür)
SOGUMA_TABANI = 15.0      # 429 gelince TÜM istekler 15 sn durur; tekrar gelirse 30, 60, 120 sn
MAKS_SOGUMA = 120.0       # Bunu aşan engelde tarama durur (sunucuyu dövmeye devam etmeyiz)
DENEME_SAYISI = 4         # 1 ilk deneme + 3 tekrar
BEKLEME_TABANI = 1.0      # 429 dışı hatalarda üstel geri çekilme: ~1, 2, 4 sn
API_BEKLEME_SURESI = 2.0  # yfinance istekleri arası en az bekleme (v6'daki hız kesici)
SAAT_DILIMI = ZoneInfo("Europe/Istanbul")

# Yalnızca yfinance'in FutureWarning kirliliğini sustur; diğer uyarılar görünür kalsın.
warnings.filterwarnings("ignore", category=FutureWarning, module="yfinance")
# websocket kütüphanesi her 429'da tüm HTML yanıtını basıyor; aynı bilgiyi kendi logumuz kısa veriyor.
logging.getLogger("websocket").setLevel(logging.CRITICAL)


def _kaydedici_kur() -> logging.Logger:
    """Colab'de basicConfig çoğu zaman etkisiz kalır; kendi işleyicimizi kuruyoruz."""
    kaydedici = logging.getLogger("bist_tarayici")
    if not kaydedici.handlers:  # hücre yeniden çalışınca satırlar ikiye katlanmasın
        isleyici = logging.StreamHandler(sys.stdout)
        bicim = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", "%H:%M:%S")
        bicim.converter = lambda t: datetime.fromtimestamp(t, SAAT_DILIMI).timetuple()
        isleyici.setFormatter(bicim)
        kaydedici.addHandler(isleyici)
    kaydedici.setLevel(logging.INFO)
    kaydedici.propagate = False
    return kaydedici


log = _kaydedici_kur()
T = TypeVar("T")


# =====================================================================
# MODÜL 1: DAYANIKLI VERİ KATMANI (hız kesici + retry)
# =====================================================================
class SunucuEngeli(RuntimeError):
    """Sunucu, soğuma süresi MAKS_SOGUMA'yı aşacak kadar ısrarla 429 veriyor."""


class TaramaIptal(RuntimeError):
    """Tarama durduruldu (engel ya da kullanıcı kesmesi); kalan istekler açılmaz."""


class GeciciEngel(RuntimeError):
    """Tek bir 429; ortak soğuma başlatıldı, istek sonra tekrar denenecek."""


class HizKesici:
    """
    Tüm iş parçacıklarının paylaştığı hız sınırı.
    - Yeni istekler arasında en az `aralik` + rastgele `sapma` saniye bırakır.
    - 429 gelince herkes birlikte bekler; ard arda engelde bekleme ikiye katlanır.
    - Bekleme MAKS_SOGUMA'yı aşarsa durdurur; kalan istekler hiç açılmaz.
    """

    def __init__(self, ad: str, aralik: float, sapma: float,
                 soguma: float = SOGUMA_TABANI, maks_soguma: float = MAKS_SOGUMA):
        self.ad, self.aralik, self.sapma = ad, aralik, sapma
        self.soguma_tabani, self.maks_soguma = soguma, maks_soguma
        self._kilit = threading.Lock()
        self.durdur = threading.Event()
        self.sifirla()

    def sifirla(self) -> None:
        with self._kilit:
            self._sonraki = 0.0
            self._soguma = 0.0
            self._soguma_bitis = 0.0
            self.engel_sayisi = 0
            self._durdurma_ani = 0.0
            self.durdur.clear()

    def durdur_et(self) -> None:
        self._durdurma_ani = time.monotonic()
        self.durdur.set()

    def bekle(self) -> None:
        """Sıradaki istek hakkını alır; gerekirse uyur (durdurulursa hemen uyanır)."""
        if self.durdur.is_set():
            if time.monotonic() - self._durdurma_ani < self.maks_soguma:
                raise TaramaIptal(f"{self.ad} durduruldu")
            self.sifirla()  # durdurmanın üzerinden yeterince zaman geçti; yeniden denemeye izin ver
        with self._kilit:
            an = max(time.monotonic(), self._sonraki)
            self._sonraki = an + self.aralik + random.uniform(0, self.sapma)
        kalan = an - time.monotonic()
        if kalan > 0 and self.durdur.wait(kalan):
            raise TaramaIptal(f"{self.ad} durduruldu")

    def basarili(self) -> None:
        with self._kilit:
            self._soguma = 0.0

    def engellendi(self) -> float:
        """429 bildirir; ortak soğumayı başlatır, süreyi döndürür."""
        with self._kilit:
            simdi = time.monotonic()
            self.engel_sayisi += 1
            if simdi >= self._soguma_bitis:  # zaten soğuyorsak eşzamanlı 429'lar süreyi katlamaz
                self._soguma = self.soguma_tabani if self._soguma == 0 else self._soguma * 2
                if self._soguma > self.maks_soguma:
                    self.durdur_et()
                    raise SunucuEngeli(
                        f"{self.ad} ısrarla 429 veriyor ({self.engel_sayisi} kez). Tarama durduruldu; "
                        f"birkaç dakika bekleyip tekrar çalıştırın ya da Colab'de çalışma zamanını "
                        f"silip yeniden bağlanın (yeni makine çoğu zaman yeni IP demektir).")
                self._soguma_bitis = simdi + self._soguma
                self._sonraki = max(self._sonraki, self._soguma_bitis)
                log.warning("%s 429 verdi: tüm istekler %.0f sn duraklatıldı.", self.ad, self._soguma)
            return self._soguma


TV_HIZ = HizKesici("TradingView", TV_ISTEK_ARALIGI, TV_SAPMA)
YF_HIZ = HizKesici("Yahoo Finance", API_BEKLEME_SURESI, 1.0)


def _engel_mi(hata: Exception) -> bool:
    metin = str(hata)
    return "429" in metin or "Too Many Requests" in metin


def yeniden_dene(islem: Callable[[], T], etiket: str,
                 deneme: int = DENEME_SAYISI, taban: float = BEKLEME_TABANI) -> T:
    """İşlemi üstel geri çekilmeyle tekrar dener; son denemede hatayı yukarı fırlatır."""
    for n in range(1, deneme + 1):
        try:
            return islem()
        except (SunucuEngeli, TaramaIptal):
            raise  # tekrar denemek anlamsız: tarama durduruldu
        except GeciciEngel as hata:
            if n == deneme:
                raise
            # Bekleme hız kesicide: sonraki istek soğuma bitene kadar kendiliğinden bekler.
            log.info("[%s] %d. deneme 429'a takıldı; soğuma sonrası tekrar denenecek.", etiket, n)
        except Exception as hata:
            if n == deneme:
                raise
            bekle = taban * 2 ** (n - 1) + random.uniform(0, taban)
            log.warning("[%s] %d. deneme başarısız (%s: %s); %.1f sn sonra tekrar.",
                        etiket, n, type(hata).__name__, str(hata)[:120], bekle)
            time.sleep(bekle)
    raise RuntimeError("erişilemez")  # deneme < 1 ise


def _kapanis_sutunu(df: pd.DataFrame) -> Optional[str]:
    for ad in ("Close", "Kapanış"):
        if ad in df.columns:
            return ad
    return None


def endeks_sembolleri(endeks: str = ENDEKS) -> list[str]:
    def _cek() -> list[str]:
        semboller = bp.Index(endeks).component_symbols
        if not semboller:
            raise ValueError(f"{endeks} bileşen listesi boş döndü.")
        return list(dict.fromkeys(semboller))  # sırayı koru, tekrarları at
    return yeniden_dene(_cek, endeks)


def gunluk_veri_cek(sembol: str) -> pd.DataFrame:
    """bp.download ile aynı kaynak (TradingView); ama hatayı yutmaz, böylece tekrar denenir."""
    def _cek() -> pd.DataFrame:
        TV_HIZ.bekle()
        try:
            df = bp.Ticker(sembol).history(period=VERI_PERIYODU)
        except Exception as hata:
            if _engel_mi(hata):
                TV_HIZ.engellendi()
                raise GeciciEngel("429 Too Many Requests") from None
            raise
        if df is None or df.empty or _kapanis_sutunu(df) is None:
            raise ValueError("boş ya da kapanış sütunu olmayan veri")
        TV_HIZ.basarili()
        return df
    return yeniden_dene(_cek, sembol)


def toplu_veri_cek(semboller: list[str], paralel: int = PARALEL_ISTEK
                   ) -> Tuple[dict[str, pd.DataFrame], dict[str, str]]:
    """
    bp.download() içeride sembolleri sırayla çeker ve hataları sessizce atlar.
    Burada istekler ortak hız kesiciden geçer (aralıklı + düzensiz, 429'da toplu soğuma),
    en fazla `paralel` bağlantı açık kalır ve alınamayan her hisse raporlanır.
    """
    veriler: dict[str, pd.DataFrame] = {}
    hatalar: dict[str, str] = {}
    if not semboller:
        return veriler, hatalar
    TV_HIZ.sifirla()
    bp.Ticker(semboller[0])  # TradingView sağlayıcı tekilini ana iş parçacığında oluştur
    havuz = ThreadPoolExecutor(max_workers=paralel, thread_name_prefix="bist")
    try:
        gorevler = {havuz.submit(gunluk_veri_cek, s): s for s in semboller}
        for i, gorev in enumerate(as_completed(gorevler), 1):
            sembol = gorevler[gorev]
            try:
                veriler[sembol] = gorev.result()
            except SunucuEngeli as hata:
                hatalar[sembol] = "sunucu engeli"
                log.error(str(hata))
            except TaramaIptal:
                hatalar[sembol] = "tarama durdurulduğu için istenmedi"
            except Exception as hata:
                hatalar[sembol] = f"{type(hata).__name__}: {str(hata)[:150]}"
                log.error("[%s] %d denemeden sonra veri alınamadı: %s",
                          sembol, DENEME_SAYISI, hatalar[sembol])
            if i % 20 == 0:
                log.info("İlerleme: %d/%d", i, len(semboller))
    except BaseException:  # Ctrl+C / "Yürütmeyi durdur": bekleyen istekleri hiç açma
        TV_HIZ.durdur_et()
        raise
    finally:
        havuz.shutdown(wait=True, cancel_futures=True)
    return veriler, hatalar


# =====================================================================
# MODÜL 2: VEKTÖREL TARAMA MOTORU
# =====================================================================
SONUC_SUTUNLARI = ["Sembol", "Fiyat (TL)", f"{SMA_KISA} SMA (TL)", f"{SMA_UZUN} SMA (TL)",
                   "Mesafe (%)", "Son Veri"]


def tarama_yap(veriler: dict[str, pd.DataFrame], sira: list[str]
               ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Tüm hisseleri tek bir uzun (Sembol, Tarih) serisinde birleştirip metrikleri
    groupby ile tek geçişte hesaplar; hisse başına Python döngüsü yok.

    v6 ile eşdeğerlik: v6 her hissede NaN kapanışları atıp tarihe göre sıralıyor ve
    son satırdaki rolling(N).mean() değerini alıyordu. Bu, son N geçerli kapanışın
    ortalamasıdır; burada groupby().tail(N).mean() ile aynısı hesaplanır.
    """
    seriler = {}
    for sembol, df in veriler.items():
        sutun = _kapanis_sutunu(df)
        if sutun is not None:
            seriler[sembol] = pd.to_numeric(df[sutun], errors="coerce")
    if not seriler:
        return pd.DataFrame(), pd.DataFrame(columns=SONUC_SUTUNLARI)

    kapanis = pd.concat(seriler)
    kapanis.index = kapanis.index.set_names(["Sembol", "Tarih"])
    kapanis = kapanis.dropna().sort_index()  # Sembol, sonra kronolojik Tarih

    grup = kapanis.groupby(level="Sembol", sort=False)
    tarihler = pd.Series(kapanis.index.get_level_values("Tarih"),
                         index=kapanis.index.get_level_values("Sembol"))
    ozet = pd.DataFrame({
        "Gun": grup.size(),
        "Fiyat": grup.last(),
        "SMA_K": grup.tail(SMA_KISA).groupby(level="Sembol", sort=False).mean(),
        "SMA_U": grup.tail(SMA_UZUN).groupby(level="Sembol", sort=False).mean(),
        "SonTarih": tarihler.groupby(level=0).max(),
    })
    # rolling(N) ile aynı davranış: N günden az veride SMA tanımsızdır (NaN).
    ozet["SMA_K"] = ozet["SMA_K"].where(ozet["Gun"] >= SMA_KISA)
    ozet["SMA_U"] = ozet["SMA_U"].where(ozet["Gun"] >= SMA_UZUN)
    ozet["Mesafe"] = (ozet["Fiyat"] - ozet["SMA_K"]) / ozet["SMA_K"].replace(0, np.nan)

    ozet["Yeterli"] = (ozet["Gun"] >= MIN_VERI_GUNU) & ozet[["SMA_K", "SMA_U", "Mesafe"]].notna().all(axis=1)
    ozet["F1"] = ozet["Yeterli"] & (ozet["SMA_K"] > ozet["SMA_U"])
    ozet["F2"] = ozet["F1"] & (ozet["Fiyat"] >= ozet["SMA_K"])
    ozet["F3"] = ozet["F2"] & (ozet["Mesafe"] <= MAKS_UZAKLIK)

    secilen = ozet[ozet["F3"]]
    secilen = secilen.loc[[s for s in sira if s in secilen.index]]  # endeks sırası (v6 gibi)
    sonuc = pd.DataFrame({
        "Sembol": secilen.index,
        "Fiyat (TL)": secilen["Fiyat"].round(2).to_numpy(),
        f"{SMA_KISA} SMA (TL)": secilen["SMA_K"].round(2).to_numpy(),
        f"{SMA_UZUN} SMA (TL)": secilen["SMA_U"].round(2).to_numpy(),
        "Mesafe (%)": (secilen["Mesafe"] * 100).round(2).to_numpy(),
        "Son Veri": secilen["SonTarih"].dt.strftime("%d.%m.%Y").to_numpy(),
    }, columns=SONUC_SUTUNLARI)
    return ozet, sonuc


def gunluk_grafik_verisi(df: pd.DataFrame) -> pd.DataFrame:
    """Yalnızca grafiği çizilecek birkaç hisse için tam SMA serileri (v6'daki 'DF')."""
    sutun = _kapanis_sutunu(df)
    cikti = pd.DataFrame(index=df.index)
    cikti["Close"] = pd.to_numeric(df[sutun], errors="coerce")
    cikti = cikti.dropna(subset=["Close"]).sort_index()
    cikti["SMA_KISA"] = cikti["Close"].rolling(window=SMA_KISA).mean()
    cikti["SMA_UZUN"] = cikti["Close"].rolling(window=SMA_UZUN).mean()
    return cikti


# =====================================================================
# MODÜL 3: SAATLİK VERİ, İNDİKATÖRLER VE ÇİZİM
# =====================================================================
def rsi_hesapla(fiyatlar: pd.Series, periyot: int = 14) -> pd.Series:
    """Wilder RSI. Kayıp 0 ise RSI 100; ilk `periyot` çubuk (ısınma) NaN kalır."""
    fark = fiyatlar.diff()
    kazanc = fark.clip(lower=0.0)
    kayip = -fark.clip(upper=0.0)
    avg_kazanc = kazanc.ewm(com=periyot - 1, min_periods=periyot).mean()
    avg_kayip = kayip.ewm(com=periyot - 1, min_periods=periyot).mean()
    rs = avg_kazanc / avg_kayip.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.mask(avg_kayip.eq(0) & avg_kazanc.notna(), 100.0)


def macd_hesapla(fiyatlar: pd.Series, kisa: int = 12, uzun: int = 26, sinyal_periyot: int = 9
                 ) -> Tuple[pd.Series, pd.Series, pd.Series]:
    kisa_ema = fiyatlar.ewm(span=kisa, adjust=False).mean()
    uzun_ema = fiyatlar.ewm(span=uzun, adjust=False).mean()
    macd_cizgisi = kisa_ema - uzun_ema
    sinyal_cizgisi = macd_cizgisi.ewm(span=sinyal_periyot, adjust=False).mean()
    return macd_cizgisi, sinyal_cizgisi, macd_cizgisi - sinyal_cizgisi


def _bos_degilse(df: Optional[pd.DataFrame]) -> pd.DataFrame:
    if df is None or df.empty:
        raise ValueError("boş veri döndü")
    return df


def _yf_ayikla(ham: Optional[pd.DataFrame], yf_sembol: str) -> pd.DataFrame:
    """yfinance çıktısından tek hisseyi düz sütunlarla çıkarır (tekli/çoklu indirme fark etmez)."""
    if ham is None or ham.empty:
        return pd.DataFrame()
    df = ham
    if isinstance(df.columns, pd.MultiIndex):
        if yf_sembol in df.columns.get_level_values(0):
            df = df[yf_sembol]
        elif yf_sembol in df.columns.get_level_values(1):
            df = df.xs(yf_sembol, axis=1, level=1)
        else:
            return pd.DataFrame()
    df = df.dropna(how="all")  # çoklu indirmede bu hisseye ait olmayan zaman satırları
    if df.empty or "Close" not in df.columns:
        return pd.DataFrame()
    return df.ffill().bfill().copy()


def saatlik_veri_cek(semboller: list[str]) -> dict[str, pd.DataFrame]:
    """
    Hedefleri v6 gibi tek tek çeker; istekler arası en az API_BEKLEME_SURESI + rastgele sapma.
    (yfinance'in toplu/çok iş parçacıklı indirmesi Yahoo'yu aynı anda yükler; kullanılmıyor.)
    """
    sonuc: dict[str, pd.DataFrame] = {}
    YF_HIZ.sifirla()
    for sembol in semboller:
        yf_sembol = f"{sembol}.IS"

        def _cek(ys: str = yf_sembol) -> pd.DataFrame:
            YF_HIZ.bekle()
            try:
                ham = yf.download(ys, period=SAATLIK_PERIYOT, interval="1h",
                                  progress=False, threads=False)
            except Exception as hata:
                if _engel_mi(hata):
                    YF_HIZ.engellendi()
                    raise GeciciEngel("429 Too Many Requests") from None
                raise
            df = _bos_degilse(_yf_ayikla(ham, ys))
            YF_HIZ.basarili()
            return df

        try:
            sonuc[sembol] = yeniden_dene(_cek, yf_sembol)
        except (SunucuEngeli, TaramaIptal) as hata:
            log.error("Saatlik veri çekimi durduruldu: %s", hata)
            break
        except Exception as hata:
            log.error("[%s] Saatlik veri alınamadı: %s", sembol, hata)
    return sonuc


def operasyon_ekrani_ciz(sembol: str, gunluk_veri: Optional[pd.DataFrame] = None,
                         saatlik_veri: Optional[pd.DataFrame] = None) -> bool:
    """
    Verilmeyen veri kendisi çekilir: günlük -> borsapy (taramayla aynı kaynak),
    saatlik -> yfinance. Tarama döngüsü ikisini de hazır verir, ek istek olmaz.
    """
    fig = None
    try:
        if gunluk_veri is None:
            gunluk_veri = gunluk_grafik_verisi(gunluk_veri_cek(sembol))
        if saatlik_veri is None:
            saatlik_veri = saatlik_veri_cek([sembol]).get(sembol)
        if saatlik_veri is None or saatlik_veri.empty:
            log.error("[%s] Saatlik veri yok; grafik çizilmedi.", sembol)
            return False

        saatlik = saatlik_veri.copy()
        saatlik["RSI"] = rsi_hesapla(saatlik["Close"])
        saatlik["MACD"], saatlik["Sinyal"], saatlik["Hist"] = macd_hesapla(saatlik["Close"])

        with plt.style.context("dark_background"):
            fig = plt.figure(figsize=(18, 10))
            fig.suptitle(f"SİSTEM: {sembol} | SIFIR GÜVEN (ZERO TRUST) TERMİNALİ",
                         fontsize=16, fontweight="bold", color="cyan")

            ax1 = plt.subplot2grid((3, 2), (0, 0), rowspan=2)
            ax1.plot(gunluk_veri.index, gunluk_veri["Close"], label="Kapanış (1G)", color="white", linewidth=1.5)
            ax1.plot(gunluk_veri.index, gunluk_veri["SMA_KISA"], label=f"{SMA_KISA} SMA (Tetik/Destek)", color="#00ff00", linewidth=2)
            ax1.plot(gunluk_veri.index, gunluk_veri["SMA_UZUN"], label=f"{SMA_UZUN} SMA (Ana Trend)", color="#ff0044", linewidth=2)
            ax1.set_title("KATMAN 2: Makro Trend ve Mesafe Analizi", color="gray")
            ax1.grid(True, alpha=0.2, linestyle="--")
            ax1.legend(loc="upper left", framealpha=0.3)

            ax2 = plt.subplot2grid((3, 2), (0, 1), rowspan=1)
            ax2.plot(saatlik.index, saatlik["Close"], label="Kapanış (1S)", color="white", linewidth=1.2)
            ax2.set_title("KATMAN 3: Saatlik Pusu (Son 1 Ay)", color="gray")
            ax2.grid(True, alpha=0.2, linestyle="--")
            ax2.legend(loc="upper left", framealpha=0.3)

            ax3 = plt.subplot2grid((3, 2), (1, 1), sharex=ax2)
            ax3.plot(saatlik.index, saatlik["MACD"], label="MACD", color="#00ccff", linewidth=1.2)
            ax3.plot(saatlik.index, saatlik["Sinyal"], label="Sinyal", color="#ffaa00", linewidth=1.2)
            # Tarih ekseninde genişlik gün cinsindendir: 0.6/24 ≈ 36 dk (varsayılan 0.8 gün üst üste biniyordu)
            ax3.bar(saatlik.index, saatlik["Hist"], width=0.6 / 24,
                    color=np.where(saatlik["Hist"] > 0, "#00ff00", "#ff0044"), alpha=0.6)
            ax3.set_title("Saatlik MACD (Momentum)", color="gray")
            ax3.grid(True, alpha=0.2, linestyle="--")
            ax3.legend(loc="upper left", framealpha=0.3)

            ax4 = plt.subplot2grid((3, 2), (2, 1), sharex=ax2)
            ax4.plot(saatlik.index, saatlik["RSI"], color="#cc00ff", label="RSI (14)", linewidth=1.5)
            ax4.axhline(70, color="#ff0044", linestyle="--", alpha=0.7, label="Aşırı Alım")
            ax4.axhline(30, color="#00ff00", linestyle="--", alpha=0.7, label="Aşırı Satım (Pusu)")
            ax4.fill_between(saatlik.index, saatlik["RSI"], 30,
                             where=(saatlik["RSI"] <= 30).to_numpy(), color="#00ff00", alpha=0.3)
            ax4.set_title("Saatlik RSI Gücü", color="gray")
            ax4.grid(True, alpha=0.2, linestyle="--")
            ax4.legend(loc="upper left", framealpha=0.3)

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        return True
    except Exception as hata:
        log.critical("[%s] Çizim çekirdeği çöktü: %s: %s", sembol, type(hata).__name__, hata)
        return False
    finally:
        if fig is not None:
            plt.close(fig)  # OOM koruması: çizilen figürü bellekten sil


log.info("Motor yüklendi: %s | SMA %d/%d | marj %%%.0f | en az %d gün | %d bağlantı, istek arası %.1f–%.1f sn",
         ENDEKS, SMA_KISA, SMA_UZUN, MAKS_UZAKLIK * 100, MIN_VERI_GUNU, PARALEL_ISTEK,
         TV_ISTEK_ARALIGI, TV_ISTEK_ARALIGI + TV_SAPMA)

In [ ]:
# =====================================================================
# OTONOM TARAMA (v7) — strateji v6 ile aynı; veri çekimi hız kesicili + dayanıklı
# =====================================================================
t_basla = time.perf_counter()
log.info("Sıfır Güven Motoru v7.0 başlatılıyor...")

bist_sembolleri = endeks_sembolleri()  # 4 denemede de alınamazsa hata verip durur (devam etmek anlamsız)
tahmini = len(bist_sembolleri) * (TV_ISTEK_ARALIGI + TV_SAPMA / 2)
log.info("%s: %d tahta çekiliyor (en fazla %d bağlantı, tahmini ~%.0f sn; sunucu 429 verirse uzar)...",
         ENDEKS, len(bist_sembolleri), PARALEL_ISTEK, tahmini)

gunluk_veriler, basarisiz = toplu_veri_cek(bist_sembolleri)
t_ag = time.perf_counter()
if not gunluk_veriler:
    raise RuntimeError("Hiçbir hissenin verisi alınamadı: ağ bağlantısını ya da TradingView erişimini kontrol edin.")

ozet, sonuc_df = tarama_yap(gunluk_veriler, bist_sembolleri)
t_hesap = time.perf_counter()

# --- Sessiz hata dedektörleri ---
kapsama = len(gunluk_veriler) / len(bist_sembolleri)
eksik_tarama = TV_HIZ.durdur.is_set() or kapsama < 0.9
if TV_HIZ.engel_sayisi:
    log.warning("TradingView bu taramada %d kez 429 verdi.", TV_HIZ.engel_sayisi)
if eksik_tarama:
    log.warning("EKSİK TARAMA: kapsama %%%.0f. Sonuç tüm endeksi temsil etmiyor.", kapsama * 100)
if not ozet.empty:
    en_guncel = ozet["SonTarih"].max().normalize()
    geride = ozet.index[ozet["SonTarih"].dt.normalize() < en_guncel].tolist()
    if geride:
        log.warning("Son işlem günü diğerlerinden eski (tahta kapalı / işlem durdurulmuş olabilir): %s",
                    ", ".join(geride))
    log.info("Filtre hunisi: veri %d -> yeterli %d -> F1 %d -> F2 %d -> F3 %d",
             len(ozet), ozet["Yeterli"].sum(), ozet["F1"].sum(), ozet["F2"].sum(), ozet["F3"].sum())

# --- SİSTEM ÇIKTISI (RAPORLAMA) ---
print("\n" + "=" * 65)
print("TARAMA SONUCU (v7.0): KUSURSUZ PUSU NOKTALARI" + ("  [EKSİK TARAMA]" if eksik_tarama else ""))
print("=" * 65)
if not sonuc_df.empty:
    print(sonuc_df.to_string(index=False))
    print("-" * 65)
    print("Operasyonel Komut: Aşağıdaki hücre seçilenlerin 1G + 1S grafiklerini çizer.")
else:
    print("Sistem Çıktısı: 0 Eşleşme.")
    print("Sıfır Güven Kalkanı: Piyasada kurallarımıza uyan güvenli varlık bulunamadı. Nakitte bekle.")
print("-" * 65)
print(f"Kapsama: {len(gunluk_veriler)}/{len(bist_sembolleri)} tahta | "
      f"Ağ: {t_ag - t_basla:.1f} sn | Hesaplama: {(t_hesap - t_ag) * 1000:.0f} ms | "
      f"Tarama anı: {datetime.now(SAAT_DILIMI):%d.%m.%Y %H:%M} (veri 15 dk gecikmeli)")
if basarisiz:
    print("Veri alınamayan tahtalar: " + ", ".join(sorted(basarisiz)))

ALTTAKİ HÜCRE, TARAMADA SEÇİLEN HİSSELERİ OTONOM OLARAK GRAFİĞE DÖKÜYOR (ANALİZ İÇİN).
Günlük veri taramadan bellekte hazır gelir (ek istek yok); saatlik veri yfinance'ten hisse hisse, en az 2 sn arayla çekilir.

In [ ]:
if sonuc_df.empty:
    print("Çizilecek hedef yok (0 eşleşme).")
else:
    hedefler = sonuc_df["Sembol"].tolist()
    t0 = time.perf_counter()
    saatlik_veriler = saatlik_veri_cek(hedefler)
    log.info("Saatlik veri: %d/%d hedef, %.1f sn", len(saatlik_veriler), len(hedefler), time.perf_counter() - t0)
    for sembol in hedefler:
        operasyon_ekrani_ciz(sembol,
                             gunluk_grafik_verisi(gunluk_veriler[sembol]),
                             saatlik_veriler.get(sembol, pd.DataFrame()))

### Tek hisse inceleme
Taramadan bağımsız herhangi bir hisseyi çizmek için sembolü değiştirin.

In [ ]:
operasyon_ekrani_ciz('ASELS')

In [ ]:
# KAP bildirimleri (Kamuyu Aydınlatma Platformu)
try:
    print(yeniden_dene(lambda: bp.Ticker("KRDMD").news, "KRDMD-KAP"))
except Exception as hata:
    log.error("KAP bildirimleri alınamadı: %s: %s", type(hata).__name__, hata)